# 01 — Global ocean surface heat fluxes

Global snapshot maps of the ocean-side surface heat flux terms on the LLC2160 grid:

- $Q_{net}$ — net surface heat flux including shortwave (`oceQnet`),
- $Q_{sw}$ — net shortwave radiation (`oceQsw`),
- $Q_{ns} = Q_{net} - Q_{sw}$ — non-solar flux (latent + sensible + net longwave).

All fields are converted to **positive downward** (positive warms the ocean), in W m$^{-2}$.
Global maps use area-weighted binning of the ~2–4 km cells onto a 0.25° lat-lon grid;
only a single time snapshot is loaded.

In [ ]:
# Environment check: this notebook must run on SciServer (Kraken domain,
# Oceanography image, "Poseidon DYAMOND (ceph)" data volume), or with
# DYAMOND_ROOT pointing at a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()  # raises with setup instructions if the data volume is absent
print(f"DYAMOND root: {root}")

In [ ]:
# Local dask cluster inside the SciServer container: keeps zarr reads parallel
# while bounding memory. Adjust n_workers to the container allocation.
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=8, threads_per_worker=2, memory_limit="8GB")
client = Client(cluster)
client

In [ ]:
from dyamond_fluxes import find_stores_with, open_store

ocean_store = next(iter(find_stores_with(["oceQnet"])))
ds = open_store(ocean_store)
print(ocean_store)

In [ ]:
# Choose a snapshot: boreal summer example. Change freely.
SNAPSHOT = "2020-07-15T12:00"
snap = ds.sel(time=SNAPSHOT, method="nearest")
print("snapshot:", snap.time.values)

In [ ]:
from dyamond_fluxes import nonsolar_flux, to_positive_down

# to_positive_down infers the sign convention from variable attributes and
# raises if ambiguous — in that case set assume_upward explicitly using the
# convention recorded in notebook 00.
qnet = to_positive_down(snap["oceQnet"])
qsw = to_positive_down(snap["oceQsw"])
qns = nonsolar_flux(qnet, qsw)

# Land/ice mask: MITgcm writes exact zeros over land; mask where depth is zero.
if "Depth" in ds:
    ocean_mask = ds["Depth"] > 0
    qnet, qsw, qns = (q.where(ocean_mask) for q in (qnet, qsw, qns))

In [ ]:
from pathlib import Path

from dyamond_fluxes.plotting import plot_global

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

lon, lat, area = ds["XC"], ds["YC"], ds.get("rA")

fig, ax, qnet_binned = plot_global(
    qnet.load(), lon, lat, area=area,
    title=f"Net surface heat flux, {str(snap.time.values)[:16]}",
)
fig.savefig(FIGDIR / "qnet_global.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax, _ = plot_global(
    qsw.load(), lon, lat, area=area, diverging=False,
    title=f"Net shortwave radiation, {str(snap.time.values)[:16]}",
)
fig.savefig(FIGDIR / "qsw_global.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax, _ = plot_global(
    qns.load(), lon, lat, area=area,
    title=f"Non-solar heat flux (latent + sensible + net LW), {str(snap.time.values)[:16]}",
)
fig.savefig(FIGDIR / "qns_global.png", dpi=200, bbox_inches="tight")

In [ ]:
from dyamond_fluxes import area_weighted_mean

if area is not None:
    for name, q in [("Qnet", qnet), ("Qsw", qsw), ("Qns", qns)]:
        value = float(area_weighted_mean(q, area.where(ocean_mask)).compute())
        print(f"global ocean-mean {name}: {value:8.2f} W m-2")

The instantaneous global-mean $Q_{net}$ reflects the diurnal/seasonal phase of the snapshot,
not the climatological imbalance; a time-mean over the 14-month record (dask reduction over
`time`) is the natural next step.